# Foundry IQ knowledge base with Fabric IQ integration

This notebook creates a Foundry IQ knowledge base with a **Fabric Ontology Knowledge Source** that connects to structured operational data in Microsoft Fabric.

- **Fabric Ontology Knowledge Source**: connects to a Fabric workspace ontology. The KB issues structured queries against entity types like `Product` at runtime.
- **Delegated token**: This notebook uses `InteractiveBrowserCredential` to sign in the current user and obtain a delegated token for Azure AI Search. Search uses that delegated user identity when querying protected sources such as Fabric Ontology.

## Step 1: Set up environment variables

Load the configuration for your Azure AI Search, Azure OpenAI, and Fabric resources. The `.env` file in the project folder has the values needed for this part.

In [1]:
import json
import os

from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureDeveloperCliCredential(tenant_id=AZURE_TENANT_ID)
print("Environment variables loaded")


Environment variables loaded


## Step 2: Login and acquire delegated token

In order for the knowledge base retrieval call to use your identity for query source access, it needs an OAuth token for a signed-in user who has access to the Fabric workspace.

Run the cell below to sign in to Azure using the credentials in the instructions sidebar. If successful, it acquires a delegated token for the Azure AI Search scope and saves it for use as `query_source_authorization` during retrieval.

In [2]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

Acquired token for logged-in user


## Step 3: Create three knowledge sources

For this knowledge base, you'll create three knowledge sources: the same two search-index sources used earlier, plus a Fabric Ontology knowledge source.

* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `fabric-ontology-knowledge-source`: Points to the Fabric workspace and ontology used for structured operational data

In [3]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Contoso Fabric Ontology knowledge source with product data",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)
print("Knowledge sources created")


Knowledge sources created


## Step 4: Create the multi-source + Fabric knowledge base

A knowledge base combines the knowledge sources with an LLM and instructions for how retrieval should behave.

For this notebook, the knowledge base uses an `outputMode` of `answerSynthesis` so the attached model can answer the question directly. It also uses a `retrievalReasoningEffort` of `low` so the model can help with query planning and source selection.

In [4]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-fabric-ontology-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining indexed company documents and Contoso structured product data",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Fabric Ontology for structured operational data. Use the search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


Knowledge base created


## Step 5: Query the knowledge base

Ask a question that combines structured Fabric Ontology data with reasoning from the attached chat model.

The knowledge base uses agentic retrieval to decide when to query the search indexes and when to query the Fabric Ontology source.

⏱️ The retrieval takes ~40 seconds to complete, due to the increased latency from the Fabric IQ knowledge source.

In [5]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("For the summer sale, which product categories currently have the lowest stock levels? "
            "Also, from HR documents, which job roles handle inventory strategy and budget oversight?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))


The product categories with the lowest current stock levels are **Hand Tools** at **11,207** units[ref_id:1], **Plumbing** at **14,477** units[ref_id:1], **Garden & Outdoor** at **14,492** units[ref_id:1], **Lumber & Building Materials** at **16,008** units[ref_id:1], and **Electrical** at **16,485** units[ref_id:1].

From the HR role documents, the role that most directly handles **inventory strategy** and **budget oversight** is the **Director of Operations**[ref_id:0][ref_id:21]. The description states this role will **develop and implement strategies to manage inventory and stock levels**[ref_id:0][ref_id:21] and **manage the budget and financial operations of the company**[ref_id:0].

Other HR roles tied to **budget oversight** in operations, though not explicitly to inventory strategy, include the **Vice President of Operations**, which **oversees the development and implementation of operational plans and budgets**[ref_id:7], the **Senior Manager of Operations**, which **develops, implements, and monitors operational plans, budgets, and performance goals**[ref_id:8][ref_id:14], the **Manager of Operations**, which **develops and manages operational budgets and monitors financial performance**[ref_id:9], and the **Chief Operating Officer**, which **manages the budget and ensures that the organization is operating within its means**[ref_id:6].

### Review the activity log

For this activity log, you will see a "fabricOntology" step for the call made to Fabric IQ, along with "searchIndex" steps for each query sent to a search index.

In [6]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "startedAt": "2026-10-14T17:48:13.1443533Z",
    "completedAt": "2026-10-14T17:48:15.5021257Z",
    "inputTokens": 1486,
    "outputTokens": 103,
    "elapsedMs": 2353,
    "model": {
      "modelName": "gpt-5.4",
      "deploymentId": "gpt-5.4"
    }
  },
  {
    "type": "searchIndex",
    "id": 2,
    "startedAt": "2026-10-14T17:48:15.5237213Z",
    "completedAt": "2026-10-14T17:48:16.4624678Z",
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-10-14T17:48:15.838501Z",
    "count": 30,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "inventory strategy role budget oversight",
      "filter": null,
      "semanticConfigurationName": "semantic-configuration",
      "sourceDataFields": [
        {
          "name": "snippet"
        },
        {
          "name": "uid"
        },
        {
          "name": "snippet_parent_id"
        },
        {
          "name": "blob_path"
        }

### Review the references

For the Fabric IQ knowledge source, the references have `type: "fabricOntology"`. The `sourceData` contains `fabricAnswer` with a synthesized answer and `fabricRawData` with structured data results.

In [7]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

[
  {
    "type": "searchIndex",
    "id": "0",
    "activitySource": 3,
    "rerankerScore": 3.7473414,
    "docKey": "810256edf976_aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1_pages_14",
    "citationUrl": "https://search-r3xqdndjhq3yc.search.windows.net/indexes/hrdocs/docs/810256edf976_aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1_pages_14?$select=snippet%2Cuid%2Csnippet_parent_id%2Cblob_path&api-version=2026-08-01-preview&allowUnsafeKeys=true",
    "sourceData": {
      "uid": "810256edf976_aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1_pages_14",
      "snippet_parent_id": "aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1",
      "blob_path": "/hrdocs/role_library.pdf",
      "snippet": "written and verbal communication skills \n\n\u2022 Ability to manage and motivate a team \n\n\u20